In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from utils.utils import load_encrypted_xlsx

In [ ]:
registry_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx'
outcome_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/follow_up/aSAH_DATA_2009_2024_18122024.xlsx'
bp_path = "/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/Transfer Urs.pietsch@kssg.ch 22.01.24, 15_34/20240116_SAH_SOS_Blutdruecke.csv"
registry_pdms_correspondence_path = "/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/registry_pdms_correspondence.csv"

In [ ]:
registry_df = load_encrypted_xlsx(registry_path)
bp_df = pd.read_csv(bp_path, sep=';', decimal='.')
registry_pdms_correspondence_df = pd.read_csv(registry_pdms_correspondence_path)

In [ ]:
outcome_df = load_encrypted_xlsx(outcome_data_path)

In [ ]:
bp_df = bp_df.merge(registry_pdms_correspondence_df, how="left", on="pNr")

In [ ]:
outcome_df["mRS_FU_1y_int"] = pd.to_numeric(outcome_df["mRS_FU_1y"], errors="coerce")

In [ ]:
bp_df.head()

In [ ]:
outcome_df

In [ ]:
bp_df["Date_birth"] = pd.to_datetime(bp_df["Date_birth"], format="%Y-%m-%d")

In [ ]:
outcome_df["Date_birth"] = pd.to_datetime(outcome_df["Date_birth"])

In [ ]:
bp_df["mrs_1y"] = np.nan

In [ ]:
for pnr in tqdm(bp_df["pNr"].unique()):
    sos_center_nr = bp_df[bp_df["pNr"] == pnr]["SOS-CENTER-YEAR-NO."].values[0]
    name = bp_df[bp_df["pNr"] == pnr]["JoinedName"].values[0]
    date_birth = bp_df[bp_df["pNr"] == pnr]["Date_birth"].values[0]
    mrs_values = outcome_df[(outcome_df["SOS-CENTER-YEAR-NO."] == sos_center_nr) &
                        (outcome_df["Name"] == name) &
                        (outcome_df["Date_birth"] == date_birth)]["mRS_FU_1y_int"]
    if len(mrs_values) == 0:
        mrs = np.nan
    else:
        mrs = mrs_values.values[0]

    bp_df.loc[bp_df["pNr"] == pnr, "mrs_1y"] = mrs



In [ ]:
bp_df.head()

In [ ]:
# plot boxplot per mrs
# 3 subplots: diastole, systole, mean
fig, ax = plt.subplots(3, 1, figsize=(10, 10))

sns.boxplot(x="mrs_1y", y="systole", data=bp_df, hue="mrs_1y", ax=ax[0], legend=False)
sns.boxplot(x="mrs_1y", y="diastole", data=bp_df, hue="mrs_1y", ax=ax[1], legend=False)
sns.boxplot(x="mrs_1y", y="mitteldruck", data=bp_df, hue="mrs_1y", ax=ax[2], legend=False)

plt.show()